In [ ]:
from brainspace.gradient.utils import dominant_set
from brainspace.mesh.mesh_io import read_surface
from brainspace.utils.parcellation import map_to_labels
from matplotlib.colors import ListedColormap
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.stats import pearsonr
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multitest import fdrcorrection
from surfplot import Plot

import bct
import cmasher as cmr
import glob
import matplotlib.pyplot as plt
import natsort
import nibabel as nib
import numpy as np
import os
import pandas as pd
import pickle
import seaborn as sns
import statsmodels.formula.api as smf
import SUITPy.flatmap as flatmap

def get_files(pattern, force_list=False):
  if isinstance(pattern, list):
    pattern = os.path.join(*pattern)
  files = natsort.natsorted(glob.glob(pattern))
  if not files:
    raise FileNotFoundError('Pattern could not detect file(s)')
  if (not force_list) & (len(files) == 1):
    return files[0]
  else:
    return files

def parse_roi_names(df, col='roi'):
    cerebellum_labels = ["M1L", "M2L", "M3L", "M4L", "A1L", "A2L", "A3L", "D1L", "D2L", "D3L", "D4L",
                         "S1L", "S2L", "S3L", "S4L", "S5L", "M1R", "M2R", "M3R", "M4R", "A1R", "A2R", 
                         "A3R", "D1R", "D2R", "D3R", "D4R", "S1R", "S2R", "S3R", "S4R", "S5R"]
    cerebellum_map = {f"region{i+1}": lab for i, lab in enumerate(cerebellum_labels)}
    cerebellum_index = {f"region{i+1}": i+1 for i in range(32)}  

    subcortex_order = ["aHIP-rh","pHIP-rh","lAMY-rh","mAMY-rh","THA-DP-rh","THA-VP-rh","THA-VA-rh","THA-DA-rh",
                       "NAc-shell-rh","NAc-core-rh","pGP-rh","aGP-rh","aPUT-rh","pPUT-rh","aCAU-rh","pCAU-rh",
                       "aHIP-lh","pHIP-lh","lAMY-lh","mAMY-lh","THA-DP-lh","THA-VP-lh","THA-VA-lh","THA-DA-lh",
                       "NAc-shell-lh","NAc-core-lh","pGP-lh","aGP-lh","aPUT-lh","pPUT-lh","aCAU-lh","pCAU-lh"]
    subcortex_index = {lab: i+1 for i, lab in enumerate(subcortex_order)}  

    def parse_single_roi(roi_value):
        if pd.isna(roi_value):
            return pd.Series([None, None, None, None], index=['hemi','network','name','roi_renamed'])
        roi_str = str(roi_value).strip()

        if roi_str in cerebellum_map:
            label = cerebellum_map[roi_str]      
            hemi = "RH" if label.endswith("R") else "LH"
            network = "cerebellum"
            name_idx = cerebellum_index[roi_str]     
            return pd.Series([hemi, network, name_idx, label], index=['hemi','network','name','roi_renamed'])

        if roi_str in subcortex_index:
            hemi = "RH" if roi_str.endswith('-rh') else "LH"
            network = "subcortex"
            name_idx = subcortex_index[roi_str]    
            return pd.Series([hemi, network, name_idx, roi_str], index=['hemi','network','name','roi_renamed'])

        parts = roi_str.split('_', 3)
        if len(parts) == 4:
            hemi, network, name = parts[1], parts[2], parts[3]
        else:
            hemi, network, name = None, None, roi_str
        return pd.Series([hemi, network, name, roi_str], index=['hemi','network','name','roi_renamed'])

    parsed = df[col].apply(parse_single_roi)
    df[['hemi','network','name']] = parsed[['hemi','network','name']]
    df[col] = parsed['roi_renamed']
    return df
    
def load_gradients(fname, k=None):
    df = pd.read_table(fname, index_col=0)
    if k is not None:
        df = df.iloc[:, :k]
    df = df.reset_index().rename(columns={'index': 'roi'})
    df = parse_roi_names(df)
    return df
    
def _read_gradients(fname, k, sub, task, epoch):
    df = load_gradients(fname, k)
    for col, val in zip(['sub', 'task', 'epoch'], [sub, task, epoch]):
        df[col] = val
    df['roi_ix'] = np.arange(464) + 1
    return df
    
def load_subject_gradients():
    task_files = get_files(["../output/pca-gradients-centered", "*/*_gradient.tsv"])
    subject_gradients = []
    for g in task_files:
        sub, _, _, _, epoch, _ = os.path.basename(g).split('_')
        task = "rl" if "RL" in g else "ml" if "right" in g else "unknown"
        subject_gradients.append(_read_gradients(g, 5, sub, task, epoch))
    subject_gradients = pd.concat(subject_gradients)
    return subject_gradients

def compute_eccentricity(data):
    def _get_dists(x):
        grads = x.filter(like='g')
        centroid = grads.mean().values
        x['distance'] = np.linalg.norm(grads.values - centroid, axis=1)
        return x
    return data.groupby(['sub', 'task', 'epoch']).apply(_get_dists)

def plot_subcortex(data, cmap, color_range, save_dir, surf_file='../utils/atlases/subcorticalSurf.pkl'):
    with open(surf_file, 'rb') as f:
        VL, VR = pickle.load(f)

    nROI = len(VL)

    cmap = cmap.copy()
    cmap.set_bad(color='grey')   
    cmap.set_under(color='grey')

    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(141, projection='3d') 
    ax2 = fig.add_subplot(142, projection='3d')  
    ax3 = fig.add_subplot(143, projection='3d') 
    ax4 = fig.add_subplot(144, projection='3d')  

    vmin, vmax = color_range
    def val_to_color(val):
        if val is None or np.isnan(val):
            return cmap(np.nan)
        frac = (val - vmin) / float(vmax - vmin)
        return cmap(frac)

    coords_ax1, coords_ax2, coords_ax3, coords_ax4 = [], [], [], []

    for p in range(nROI):
        if VL[p]['simplices'].size > 0:
            coords = VL[p]['coords']
            faces = VL[p]['simplices']
            color_val = data[p + nROI]  
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax1.add_collection3d(mesh)
            coords_ax1.append(coords)

    ax1.view_init(elev=0, azim=0)
    ax1.set_box_aspect((1, 1, 1))
    ax1.set_axis_off()

    for p in range(nROI):
        if VL[p]['simplices'].size > 0:
            coords = VL[p]['coords']
            faces = VL[p]['simplices']
            color_val = data[p + nROI]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax2.add_collection3d(mesh)
            coords_ax2.append(coords)

    ax2.view_init(elev=0, azim=180)
    ax2.set_box_aspect((1, 1, 1))
    ax2.set_axis_off()

    for p in range(nROI):
        if VR[p]['simplices'].size > 0:
            coords = VR[p]['coords']
            faces = VR[p]['simplices']
            color_val = data[p]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax3.add_collection3d(mesh)
            coords_ax3.append(coords)

    ax3.view_init(elev=0, azim=0)
    ax3.set_box_aspect((1, 1, 1))
    ax3.set_axis_off()

    for p in range(nROI):
        if VR[p]['simplices'].size > 0:
            coords = VR[p]['coords']
            faces = VR[p]['simplices']
            color_val = data[p]
            mesh = Poly3DCollection(coords[faces], alpha=1.0)
            mesh.set_facecolor(val_to_color(color_val))
            mesh.set_edgecolor('none')
            ax4.add_collection3d(mesh)
            coords_ax4.append(coords)

    ax4.view_init(elev=0, azim=180)
    ax4.set_box_aspect((1, 1, 1))
    ax4.set_axis_off()

    if coords_ax1:
        all_coords_1 = np.concatenate(coords_ax1, axis=0)
        ax1.auto_scale_xyz(all_coords_1[:, 0], all_coords_1[:, 1], all_coords_1[:, 2])
    if coords_ax2:
        all_coords_2 = np.concatenate(coords_ax2, axis=0)
        ax2.auto_scale_xyz(all_coords_2[:, 0], all_coords_2[:, 1], all_coords_2[:, 2])
    if coords_ax3:
        all_coords_3 = np.concatenate(coords_ax3, axis=0)
        ax3.auto_scale_xyz(all_coords_3[:, 0], all_coords_3[:, 1], all_coords_3[:, 2])
    if coords_ax4:
        all_coords_4 = np.concatenate(coords_ax4, axis=0)
        ax4.auto_scale_xyz(all_coords_4[:, 0], all_coords_4[:, 1], all_coords_4[:, 2])

    plt.tight_layout()
    plt.savefig(f"{save_dir}", dpi=800, transparent=True)
    plt.close()

def plot_cerebellum(data_array, color_range, cmap, save_dir):
    label_file = '../utils/atlases/atl-NettekovenAsym32_dseg.label.gii'
    labels = nib.load(label_file)
    num_vertices = labels.darrays[0].data.shape[0]
    data = np.zeros(num_vertices)
    roi_values = data_array
    label_data = labels.darrays[0].data
    
    for i, roi_value in enumerate(roi_values):
        data[label_data == (i + 1)] = roi_value  
    cmap = cmap
    ax = flatmap.plot(data, overlay_type='func', cmap=cmap, cscale=color_range, 
                      new_figure=True, colorbar=True, bordercolor='w')
    plt.savefig(save_dir, dpi=800, transparent=True)
    plt.close()
    
def plot_cerebellum_stats(df, value_col, color_range, cmap, save_path, label_file='../utils/atlases/atl-NettekovenAsym32_dseg.label.gii'):
    cerebellum_labels = ["M1L", "M2L", "M3L", "M4L", "A1L", "A2L", "A3L", "D1L", "D2L", "D3L", "D4L", "S1L", "S2L", "S3L", "S4L", "S5L",
        "M1R", "M2R", "M3R", "M4R", "A1R", "A2R", "A3R", "D1R", "D2R", "D3R", "D4R", "S1R", "S2R", "S3R", "S4R", "S5R"]

    vmin, vmax = color_range
    missing_val = vmin - 1.0
    roi_values = np.full(len(cerebellum_labels), np.nan, dtype=float)
    roi_map = dict(zip(df["roi"].astype(str), df[value_col].astype(float)))

    for i, label in enumerate(cerebellum_labels):
        if label in roi_map:
            roi_values[i] = roi_map[label]

    labels = nib.load(label_file)
    label_data = labels.darrays[0].data.astype(int)
    num_vertices = label_data.shape[0]
    data = np.full(num_vertices, missing_val, dtype=float)  

    for i, roi_value in enumerate(roi_values):
        mask = (label_data == (i + 1))
        if not np.any(mask):
            continue

        if np.isnan(roi_value):
            data[mask] = missing_val
        else:
            data[mask] = roi_value

    cmap = cmap.copy()
    cmap.set_under('grey')

    ax = flatmap.plot(data, overlay_type='func', cmap=cmap, cscale=(vmin, vmax), new_figure=True, colorbar=True, bordercolor='w')
    plt.savefig(save_path, dpi=800, transparent=True)
    plt.close()

def _align_labels_to_atlas(x, source_labels, target_labels):
    target = np.unique(target_labels)[1:]
    df1 = pd.DataFrame(target, index=target)
    df2 = pd.DataFrame(x, index=source_labels)
    return pd.concat([df1, df2], axis=1).iloc[:, 1:].values
    
def weights_to_vertices(data, target, labels=None):
    if isinstance(target, str): 
        vertices = nib.load(target).get_fdata().ravel()
    else:
        vertices = target.ravel()
    if labels is not None:
        data = _align_labels_to_atlas(data, labels, vertices)
    mask = vertices != 0
    map_args = dict(target_lab=vertices, mask=mask, fill=np.nan)
    if (len(data.shape) == 1) or (data.shape[1] == 1):
        weights = map_to_labels(data.ravel(),  **map_args)
    else:
        weights = [map_to_labels(x, **map_args) for x in data.T]
    return weights


def get_surfaces(style='inflated', load=True):
    surf_path = '../utils/surfaces'
    surfaces = get_files([surf_path, f'*.{style}_*'])
    if load:
        surfs = [read_surface(i) for i in surfaces]
        return dict(zip(['lh', 'rh'], surfs))
    else:
        return surfaces

def plot_cbar(cmap, vmin, vmax, orientation='vertical', size=None, n_ticks=2, 
              decimals=2, fontsize=12, show_outline=False, as_int=False):
    if size is None and (orientation == 'vertical'):
        size = (.3, 4)
    if size is None and (orientation == 'horizontal'):
        size = (4, .3)
    x = np.array([[0,1]])
    plt.figure(figsize=size)
    img = plt.imshow(x, cmap=cmap)
    plt.gca().set_visible(False)
    cax = plt.axes([0.1, 0.2, 0.8, 0.6])

    ticks = np.linspace(0, 1, n_ticks)
    tick_labels = np.around(np.linspace(vmin, vmax, n_ticks), decimals)
    cbar = plt.colorbar(orientation=orientation, cax=cax, ticks=ticks)
    if as_int:
        cbar.set_ticklabels(tick_labels.astype(int))
    else:
        cbar.set_ticklabels(tick_labels)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    
    if not show_outline:
        cbar.outline.set_visible(False)
        cbar.ax.tick_params(size=0)
    return cbar

def plot_ecc():
    os.makedirs("../figures/eccentricity", exist_ok=True)
    
    df = load_gradients('../output/pca-gradients-centered/reference_gradient.tsv', 5)
    grads = df.filter(like='g').values
    
    centroid = np.mean(grads, axis=0)
    ecc = np.linalg.norm(grads - centroid, axis=1)
    
    vmax = np.nanmax(ecc)
    vmin = np.nanmin(ecc)
    cmap = cmr.get_sub_cmap('viridis', 0.01, 0.99)
    norm = Normalize(vmin=vmin, vmax=vmax)

    cerebellum_ecc = ecc[:32] 
    subcortex_ecc = ecc[32:64] 
    cortex_ecc = ecc[64:] 

    plot_cbar(cmap, vmin, vmax, 'horizontal', size=(1, .3), n_ticks=3)
    plt.savefig("../figures/eccentricity/eccentricity_cbar.png", dpi=800, transparent=True)
    
    surfaces = get_surfaces()
    x = weights_to_vertices(cortex_ecc, "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii")
    p = Plot(surfaces['lh'], surfaces['rh'])
    p.add_layer(x, cmap=cmap, color_range=(vmin, vmax), cbar=False)
    fig = p.build(colorbar=False)
    fig.savefig("../figures/eccentricity/brain.png", dpi=800, transparent=True)
    plt.close(fig)
    plot_cerebellum(cerebellum_ecc, [vmin, vmax], cmr.get_sub_cmap('viridis', 0.01, 0.99), "../figures/eccentricity/cerebellum.png") 
    plot_subcortex(subcortex_ecc, cmr.get_sub_cmap('viridis', 0.01, 0.99), [vmin, vmax], "../figures/eccentricity/subcortex.png")

    df["ecc"] = ecc
    pairs = [("g1", "g2", "PC1", "PC2"), ("g1", "g3", "PC1", "PC3"), ("g1", "g4", "PC1", "PC4"), ("g1", "g5", "PC1", "PC5"),
             ("g2", "g3", "PC2", "PC3"), ("g2", "g4", "PC2", "PC4"), ("g2", "g5", "PC2", "PC5"), ("g3", "g4", "PC3", "PC4"),
             ("g3", "g5", "PC3", "PC5"), ("g4", "g5", "PC4", "PC5")]

    n_plots = len(pairs)
    n_cols = 5
    n_rows = int(np.ceil(n_plots / n_cols))
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3*n_cols, 3*n_rows))
    axes = axes.flatten()

    sc = None
    for ax, (xcol, ycol, xl, yl) in zip(axes, pairs):
        sc = ax.scatter(df[xcol], df[ycol], c=df["ecc"], cmap=cmap, norm=norm, s=14, edgecolor='none')
        ax.set_xlabel(xl)
        ax.set_ylabel(yl)
        ax.grid(False)
        ax.set_aspect('equal', adjustable='box')
        ax.margins(0)
        ax.use_sticky_edges = False

    for ax in axes[len(pairs):]:
        ax.axis('off')
    
    for ax in axes[:len(pairs)]:
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.autoscale(False)
    
    plt.tight_layout()
    plt.savefig("../figures/eccentricity/gradients.png", dpi=800, transparent=True)
    plt.close()

def plot_cbar(cmap, vmin, vmax, orientation='vertical', size=None, n_ticks=2, decimals=2, fontsize=12, show_outline=False, as_int=False):
    if size is None and (orientation == 'vertical'):
        size = (.3, 4)
    if size is None and (orientation == 'horizontal'):
        size = (4, .3)
    x = np.array([[0,1]])
    plt.figure(figsize=size)
    img = plt.imshow(x, cmap=cmap)
    plt.gca().set_visible(False)
    cax = plt.axes([0.1, 0.2, 0.8, 0.6])

    ticks = np.linspace(0, 1, n_ticks)
    tick_labels = np.around(np.linspace(vmin, vmax, n_ticks), decimals)
    cbar = plt.colorbar(orientation=orientation, cax=cax, ticks=ticks)
    if as_int:
        cbar.set_ticklabels(tick_labels.astype(int))
    else:
        cbar.set_ticklabels(tick_labels)
    plt.xticks(fontsize=fontsize)
    plt.yticks(fontsize=fontsize)
    
    if not show_outline:
        cbar.outline.set_visible(False)
        cbar.ax.tick_params(size=0)
    return cbar
    
def plot_3d(x, y, z, color=None, ax=None, view_3d=(35, -110), **kwargs):
    if ax is None:
        fig = plt.figure(figsize=(4, 4), frameon=False)
        ax = fig.add_subplot(projection='3d')
    
    ax.scatter(xs=x, ys=y, zs=z, c=color, **kwargs)
    ax.set(xlabel='PC1', ylabel='PC2', zlabel='PC3')
    if view_3d is not None:
        ax.view_init(elev=view_3d[0], azim=view_3d[1])
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.grid(False)
    return ax
    
def reference_eccentricity(k=5, view_3d=(20, -150)):
    df = load_gradients('../output/pca-gradients-centered/reference_gradient.tsv', 5)
    grads = df.filter(like='g').values

    centroid = np.mean(grads, axis=0)
    assert np.allclose(np.around(centroid, decimals=16), np.zeros(k))

    ecc = np.linalg.norm(grads - centroid, axis=1)
    vmax = np.nanmax(ecc)
    vmin = np.nanmin(ecc)
    cmap = 'viridis'

    fig = plt.figure(figsize=(4, 4))
    ax = fig.add_subplot(111, projection='3d')
    plot_3d(grads[:, 0], grads[:, 1], grads[:, 2], color=ecc, ax=ax, cmap=cmap, lw=0, s=10, view_3d=view_3d)
    fig.savefig("../figures/eccentricity/ecc_scatter_g1.png", dpi=800, transparent=True)
    plt.close(fig)

    plot_cbar(cmap, vmin, vmax, size=(.2, 1), fontsize=8)
    plt.savefig("../figures/eccentricity/ecc_scatter_cmap.png", dpi=800, transparent=True)
    plt.close()

    fig = plt.figure(figsize=(4, 4))
    ax = fig.add_subplot(111, projection='3d')
    plot_3d(grads[:, 4], grads[:, 3], grads[:, 2], color=ecc, ax=ax, cmap=cmap, lw=0, s=10, view_3d=view_3d)
    fig.savefig("../figures/eccentricity/ecc_scatter_g3.png", dpi=800, transparent=True)
    plt.close(fig)

    return pd.DataFrame({'roi': df['roi'], 'distance': ecc})

def plot_eccentricity_calculation():
    df = load_gradients("../output/pca-gradients-centered/reference_gradient.tsv", 5)
    schaefer_rois    = pd.read_csv("../utils/atlases/schaefer-parcellation.csv")["name_7networks"].to_list()
    cerebellum_rois  = ["cerebellum"] * 32
    subcortical_rois = ["subcortex"] * 32
    df["roi"]        = cerebellum_rois + subcortical_rois + schaefer_rois
    df["network"]    = df["roi"].apply(extract_network)
    cmap             = yeo_cmap()
    df["c"]          = df["network"].apply(lambda x: cmap[x]) 

    grads = df.filter(like="g").values
    df["ecc"] = np.linalg.norm(grads, axis=1)

    pairs = [("g1", "g2", "PC1", "PC2"), ("g1", "g3", "PC1", "PC3"), ("g1", "g4", "PC1", "PC4"), ("g1", "g5", "PC1", "PC5"),
             ("g2", "g3", "PC2", "PC3"), ("g2", "g4", "PC2", "PC4"), ("g2", "g5", "PC2", "PC5"), ("g3", "g4", "PC3", "PC4"),
             ("g3", "g5", "PC3", "PC5"), ("g4", "g5", "PC4", "PC5"),]

    fig, axes = plt.subplots(2, 5, figsize=(15, 6), sharex=True, sharey=True, constrained_layout=True)
    axes = axes.flatten()

    roi_ixs = [i for i in (10, 40, 100, 440) if 0 <= i < len(df)]
    hi = df.loc[roi_ixs].copy() if len(roi_ixs) else None

    for ax, (xcol, ycol, xl, yl) in zip(axes, pairs):
        ax.scatter(df[xcol], df[ycol], c=df["c"].to_list(), s=12, edgecolor="none", alpha=0.25)
        ax.scatter([0], [0], color="k", marker="s", s=30, zorder=6)

        if hi is not None and len(hi):
            for _, r in hi.iterrows():
                px, py = r[xcol], r[ycol]
                ax.plot([0, px], [0, py], c="k", ls="--", lw=0.8, alpha=0.9, zorder=5)
                ax.scatter([px], [py], c=[r["c"]], s=30, edgecolor="k", linewidth=0.7, zorder=7)

        ax.set_xlabel(xl)
        ax.set_ylabel(yl)
        ax.set_aspect("equal", adjustable="box")
        ax.grid(False)
        ax.set_xlim(*(-4, 4))
        ax.set_ylim(*(-4, 4))
        ax.autoscale(False)

    plt.savefig("../figures/eccentricity/eccentricity_calculation.png", dpi=800, transparent=True)
    plt.close(fig)
    
def parse_roi_labels(index):
    hemis = []
    networks = []
    rois = []

    for i, label in enumerate(index):
        hemi = None
        network = None
        roi = None
        if label.startswith("region"):
            roi = label
            num_str = label.replace("region", "")
            try:
                num = int(num_str)
            except ValueError:
                num = None
            if num is not None:
                hemi = "LH" if num <= 16 else "RH"
            else:
                hemi = "LH" if i < 16 else "RH"
            network = "cerebellum"
        elif label.endswith("-rh") or label.endswith("-lh"):
            base, hemi_suffix = label.rsplit("-", 1)
            roi = label
            hemi = "RH" if hemi_suffix == "rh" else "LH"
            network = "subcortex"
        else:
            parts = label.split("_", 3)
            if len(parts) >= 3:
                hemi = parts[1]
                network = parts[2]
            roi = label

        hemis.append(hemi)
        networks.append(network)
        rois.append(roi)
    regions = pd.DataFrame({"roi": rois, "hemi": hemis, "network": networks}, index=index)
    return regions

def correlate_measures(data):
    measures = ['strength', 'participation', 'module_degree']
    correlations = []
    y = pd.to_numeric(data['distance'], errors='coerce').to_numpy(dtype=float)
    for m in measures:
        x = pd.to_numeric(data[m], errors='coerce').to_numpy(dtype=float)
        mask = ~np.isnan(x) & ~np.isnan(y)
        if mask.sum() < 2:
            r, p = np.nan, np.nan
        else:
            r, p = pearsonr(x[mask], y[mask])
        correlations.append({'measure': m, 'r': r, 'p': p})
    correlations = pd.DataFrame(correlations)
    return correlations

def extract_network(roi):
    if roi.startswith('7Networks_'):
        return roi.split('_')[2] 
    elif '*' in roi:
        return roi.split('*')[1] 
    else:
        return roi  

def yeo_cmap():
    cmap = {'cerebellum': (0, 0, 0), 'subcortex': (77, 77, 77), 'Vis': (119, 20, 140), 'SomMot': (70, 126, 175), 'DorsAttn': (0, 117, 7), 
            'SalVentAttn': (195, 59, 255), 'Limbic': (219, 249, 165), 'Cont': (230, 149, 33), 'Default': (205, 65, 80)}
    cmap = {k: np.array(v) / 255 for k, v in cmap.items()}
    return cmap
    
def plot_correlations(data):
    schaefer_rois = pd.read_csv("../utils/atlases/schaefer-parcellation.csv")["name_7networks"].to_list()
    cerebellum_rois = ["cerebellum"] * 32
    subcortical_rois = ["subcortex"] * 32
    roi_list = cerebellum_rois + subcortical_rois + schaefer_rois

    data = data.copy()
    data["roi"] = roi_list

    data["network"] = data["roi"].apply(extract_network)
    cmap = yeo_cmap()
    data["c"] = data["network"].map(cmap)
    corrs = correlate_measures(data)
    print(corrs)

    measures = ["strength", "participation", "module_degree"]
    xlabels = ["Node strength\n", "Participation\ncoefficient", "Within-module\ndegree zscore"]

    for i, m in enumerate(measures):
        scatter_kws = dict(s=10, alpha=0.7, linewidth=0.25, edgecolor="black", clip_on=False)
        line_kws = dict(color="k", lw=1)
        fig, ax = plt.subplots(figsize=(1.2, 2))
        ax = sns.regplot(x=m, y="distance", data=data, scatter_kws=scatter_kws, line_kws=line_kws,
                         ax=ax, ci=None, color=None)
        for spine in ax.spines.values():
            spine.set_linewidth(0.5)
        ax.collections[0].set_facecolors(data["c"].values)
        ylim = (np.floor(data["distance"].min()), np.ceil(data["distance"].max()))
        xlim = (np.around(data[m].min(), 3), np.around(data[m].max(), 3))
        ax.set(xlabel=xlabels[i], ylabel="Eccentricity", ylim=ylim)
        ax.set_yticks(np.arange(0, 6))
        ax.set_ylim(0, 5)
        for spine in ax.spines.values():
            spine.set_linewidth(0.3)
        ax.tick_params(axis="both", which="major", width=0.3, length=2.5, labelsize=7)
        sns.despine()
        plt.tight_layout()
        plt.savefig(f"../figures/eccentricity/{m}.png", dpi=800, transparent=True)
        plt.close()
        
def compute_measures(cmat, thresh=.9):
    regions = parse_roi_labels(cmat.index)
    network_aff = regions["network"].values
    network_hemi_aff = (regions["hemi"] + "_" + regions["network"]).values

    x = dominant_set(cmat.values, k=1 - thresh, is_thresh=False, as_sparse=False)
    x_bin = (x != 0).astype(float)

    measures = pd.DataFrame({"roi": regions["roi"].values, "hemi": regions["hemi"].values,
               "network": regions["network"].values, "participation": bct.participation_coef(x, network_aff, "out"),
               "participation_h": bct.participation_coef(x, network_hemi_aff, "out"), "module_degree": bct.module_degree_zscore(x_bin, network_aff, 2),
               "module_degree_h": bct.module_degree_zscore(x_bin, network_hemi_aff, 2), "strength": np.sum(x, axis=1)}, index=cmat.index)
    return measures

def fdr_within_effect(results, alpha=0.05):
    results = results.copy()
    results["q"] = np.nan
    results["sig_corrected"] = 0
    for eff, sub in results.groupby("effect", sort=False):
        p = sub["p"].values
        mask = ~np.isnan(p)
        if mask.any():
            rej, qvals = fdrcorrection(p[mask], alpha=alpha)
            results.loc[sub.index[mask], "q"] = qvals
            results.loc[sub.index[mask], "sig_corrected"] = rej.astype(int)
    return results
    
def roi_run_model(df):
    rows = []
    for (roi, roi_ix), g in df.groupby(['roi', 'roi_ix'], sort=False, observed=True):
        aov = AnovaRM(g, depvar="distance", subject="sub", within=["task", "epoch"]).fit()

        for eff in aov.anova_table.index:
            rows.append({"roi": roi, "roi_ix": roi_ix, "effect": eff, "F": float(aov.anova_table.loc[eff, "F Value"]),
                         "p": float(aov.anova_table.loc[eff, "Pr > F"]), "df1": float(aov.anova_table.loc[eff, "Num DF"]),
                         "df2": float(aov.anova_table.loc[eff, "Den DF"])})

    results = pd.DataFrame(rows)
    results = fdr_within_effect(results)
    return results

def plot_test_results(roi_results):
    roi_results = roi_results[roi_results["sig_corrected"] == 1]

    #1. Plot epoch results
    roi_results_epoch = roi_results[roi_results["effect"] == "epoch"]
    e_vmin = np.min(roi_results_epoch["F"])
    e_vmax = np.max(roi_results_epoch["F"])
    print("epoch")
    print(e_vmin, e_vmax)

    #1.1 Cortex    
    cortex = roi_results_epoch[(roi_results_epoch["roi"].astype(str).str.startswith("7Net"))]
    surfaces = get_surfaces()
    cmap = cmr.get_sub_cmap('Wistia_r', 0.01, 0.99)
    x = weights_to_vertices(cortex["F"].to_numpy(), "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii", 
                            [x - 64 for x in cortex["roi_ix"]])
    p = Plot(surfaces['lh'], surfaces['rh'])
    p.add_layer(x, cmap=cmap, color_range=(e_vmin, e_vmax), cbar=False)
    p.add_layer(x, cmap='binary_r', cbar=False, as_outline=True)
    fig = p.build(colorbar=False)
    plt.savefig("../figures/stats/epoch_cortex.png", dpi=800, transparent=True)
    plt.close()
    
    #1.2 Subcortex
    subcortex = roi_results_epoch[roi_results_epoch["roi"].astype(str).str.endswith(("-lh", "-rh"))].sort_values(by="roi_ix")
    subcortex_order = ["aHIP-rh","pHIP-rh","lAMY-rh","mAMY-rh","THA-DP-rh","THA-VP-rh","THA-VA-rh","THA-DA-rh",
                       "NAc-shell-rh","NAc-core-rh","pGP-rh","aGP-rh","aPUT-rh","pPUT-rh","aCAU-rh","pCAU-rh",
                       "aHIP-lh","pHIP-lh","lAMY-lh","mAMY-lh","THA-DP-lh","THA-VP-lh","THA-VA-lh","THA-DA-lh",
                       "NAc-shell-lh","NAc-core-lh","pGP-lh","aGP-lh","aPUT-lh","pPUT-lh","aCAU-lh","pCAU-lh"]
    roi_to_idx_sc = {roi: i for i, roi in enumerate(subcortex_order)}
    f_sc = subcortex[subcortex["roi"].isin(subcortex_order)].set_index("roi")["F"]
    data_sc = np.full(len(subcortex_order), np.nan, dtype=float)
    for roi, F in f_sc.items():
        idx = roi_to_idx_sc[roi]
        if np.isfinite(F):
            data_sc[idx] = float(F)
    plot_subcortex(data_sc, cmr.get_sub_cmap('Wistia_r', 0.01, 0.99), (e_vmin, e_vmax), save_dir="../figures/stats/epoch_subcortex.png")

    #1.3 Cerebellum
    cerebellum = roi_results_epoch[roi_results_epoch["roi"].astype(str).str.endswith(("L", "R"))].sort_values(by="roi_ix")
    plot_cerebellum_stats(df=cerebellum, value_col="F", color_range=[e_vmin, e_vmax], cmap=cmr.get_sub_cmap('Wistia_r', 0.01, 0.99), save_path="../figures/stats/epoch_cerebellum.png") 

    #2. Plot task results
    roi_results_task = roi_results[roi_results["effect"] == "task"]
    t_vmin = np.min(roi_results_task["F"])
    t_vmax = np.max(roi_results_task["F"])

    print("task")
    print(t_vmin, t_vmax)
    
    #2.1 Cortex    
    cortex = roi_results_task[(roi_results_task["roi"].astype(str).str.startswith("7Net"))]
    surfaces = get_surfaces()
    cmap = cmr.get_sub_cmap('Wistia_r', 0.01, 0.99)
    x = weights_to_vertices(cortex["F"].to_numpy(), "../utils/atlases/Schaefer2018_400Parcels_7Networks_order.dlabel.nii", 
                            [x - 64 for x in cortex["roi_ix"]])
    p = Plot(surfaces['lh'], surfaces['rh'])
    p.add_layer(x, cmap=cmap, color_range=(t_vmin, t_vmax), cbar=False)
    p.add_layer(x, cmap='binary_r', cbar=False, as_outline=True)
    fig = p.build(colorbar=False)
    plt.savefig("../figures/stats/task_cortex.png", dpi=800, transparent=True)
    plt.close()

    #2.2 Subcortex
    subcortex = roi_results_task[roi_results_task["roi"].astype(str).str.endswith(("-lh", "-rh"))].sort_values(by="roi_ix")
    subcortex_order = ["aHIP-rh","pHIP-rh","lAMY-rh","mAMY-rh","THA-DP-rh","THA-VP-rh","THA-VA-rh","THA-DA-rh",
                       "NAc-shell-rh","NAc-core-rh","pGP-rh","aGP-rh","aPUT-rh","pPUT-rh","aCAU-rh","pCAU-rh",
                       "aHIP-lh","pHIP-lh","lAMY-lh","mAMY-lh","THA-DP-lh","THA-VP-lh","THA-VA-lh","THA-DA-lh",
                       "NAc-shell-lh","NAc-core-lh","pGP-lh","aGP-lh","aPUT-lh","pPUT-lh","aCAU-lh","pCAU-lh"]
    roi_to_idx_sc = {roi: i for i, roi in enumerate(subcortex_order)}
    f_sc = subcortex[subcortex["roi"].isin(subcortex_order)].set_index("roi")["F"]
    data_sc = np.full(len(subcortex_order), np.nan, dtype=float)
    for roi, F in f_sc.items():
        idx = roi_to_idx_sc[roi]
        if np.isfinite(F):
            data_sc[idx] = float(F)
    plot_subcortex(data_sc, cmr.get_sub_cmap('Wistia_r', 0.01, 0.99), (e_vmin, e_vmax), save_dir="../figures/stats/task_subcortex.png")

    #2.3 Cerebellum
    cerebellum = roi_results_task[roi_results_task["roi"].astype(str).str.endswith(("L", "R"))].sort_values(by="roi_ix")
    plot_cerebellum_stats(df=cerebellum, value_col="F", color_range=[e_vmin, e_vmax], cmap=cmr.get_sub_cmap('Wistia_r', 0.01, 0.99), save_path="../figures/stats/task_cerebellum.png") 

def plot_proportion_bars(data, comparison=None):
    order = ["subcortex", "cerebellum", "Vis", "SomMot", "DorsAttn", "Limbic", "Cont", "SalVentAttn", "Default"]
    gradients = (pd.read_csv("../output/subject_gradients.tsv", sep="\t").drop_duplicates(subset=["roi_ix"], keep="first"))

    full_count = gradients["network"].value_counts()
    res_count  = data["network"].value_counts()
    full_vals = pd.Series({net: int(full_count.get(net, 0)) for net in order})
    res_vals  = pd.Series({net: int(res_count.get(net, 0))  for net in order})

    cmap = yeo_cmap()
    colors = [cmap.get(net, (0.5, 0.5, 0.5)) for net in order]

    plt.figure(figsize=(4, 4))
    x = np.arange(len(order))
    plt.bar(x, res_vals.values, color=colors, edgecolor="black", linewidth=0.6, label="Significant ROIs")

    plt.xticks(x, order, rotation=45, ha="right")
    plt.ylabel("Number of ROIs")
    plt.xlabel("Network")
    plt.ylim(0, 70)

    plt.tight_layout()
    plt.savefig(f"../figures/stats/{comparison}_barplot.png", dpi=800, transparent=True)
    plt.close()

def main():
    os.makedirs("../figures/eccentricity", exist_ok=True)
   
    gradients = load_subject_gradients()
    gradients = compute_eccentricity(gradients)
    gradients.to_csv('../output/subject_gradients.tsv', sep='\t', index=False)
    plot_ecc()
    plot_eccentricity_calculation()
   
    ref_ecc = reference_eccentricity()
    cerebellum = [f"region{i}" for i in range(1, 33)]
    ref_ecc.loc[ref_ecc.index[:32], "roi"] = cerebellum
   
    fname = "../output/connectivity/reference_cmat.tsv"
    cmat = pd.read_table(fname, index_col=0)
    data = compute_measures(cmat)
    data = data.merge(ref_ecc, on='roi')
    plot_correlations(data)

    os.makedirs("../figures/stats", exist_ok=True)
    gradients = pd.read_csv("../output/subject_gradients.tsv", sep="\t")
    results = roi_run_model(gradients)
    results.to_csv("../output/roi_ecc_models.csv")
    plot_test_results(results)

    epoch_results = results[results["sig_corrected"] == 1]
    epoch_results = epoch_results[epoch_results["effect"] == "epoch"]
    epoch_results = parse_roi_names(epoch_results)
    epoch_results["network"] = epoch_results["network"].replace("None", np.nan).fillna("cerebellum")
    plot_proportion_bars(epoch_results, comparison="epoch")

    task_results = results[results["sig_corrected"] == 1]
    task_results = task_results[task_results["effect"] == "task"]
    task_results = parse_roi_names(task_results)
    task_results["network"] = task_results["network"].replace("None", np.nan).fillna("cerebellum")
    plot_proportion_bars(task_results, comparison="task")
    
main()